<table align="center">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visitar MIT Deep Learning</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab1/TF_Part2_Music_Generation.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png"  style="padding-bottom:5px;" />Ejecutar en Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab1/TF_Part2_Music_Generation.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png"  height="70px" style="padding-bottom:5px;"  />Ver código fuente en GitHub</a></td>
</table>

# Información de derechos de autor

In [ ]:
# Copyright 2026 MIT Introduction to Deep Learning. Todos los derechos reservados.
#
# Licenciado bajo la Licencia MIT. No puede usar este archivo excepto en cumplimiento
# con la Licencia. El uso y/o modificación de este código fuera de MIT Introduction
# to Deep Learning debe hacer referencia a:
#
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#

# Laboratorio 1: Introducción a TensorFlow y Generación de Música con RNNs

# Parte 2: Generación de Música con RNNs

En esta parte del laboratorio, exploraremos la construcción de una Red Neuronal Recurrente (RNN) para la generación de música. Entrenaremos un modelo para aprender los patrones en partituras en formato crudo usando [notación ABC](https://en.wikipedia.org/wiki/ABC_notation) y luego utilizaremos este modelo para generar nueva música.

## 2.1 Dependencias
Primero, descarguemos el repositorio del curso, instalemos las dependencias e importemos los paquetes relevantes que necesitaremos para este laboratorio.

Usaremos [Comet ML](https://www.comet.com/docs/v2/) para rastrear el desarrollo y las ejecuciones de entrenamiento de nuestro modelo. Primero, regístrate en una cuenta de Comet [en este enlace](https://www.comet.com/signup?utm_source=mit_dl&utm_medium=partner&utm_content=github
) (puedes usar tu cuenta de Google o Github). Esto generará una clave de API personal, que puedes encontrar en la primera página de 'Comenzar con Comet', en la configuración de tu cuenta, o presionando el '?' en la esquina superior derecha y luego 'Guía de inicio rápido'. Ingresa esta clave de API como la variable global `COMET_API_KEY`.

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
# POR HACER: ¡INGRESA TU CLAVE DE API AQUÍ! instrucciones arriba
COMET_API_KEY = ""

# Importar Tensorflow 2.0
import tensorflow as tf

# Descargar e importar el paquete de MIT Introduction to Deep Learning
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# Importar todos los paquetes restantes
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1


# Verificar que estamos usando una GPU, si no, cambiar el entorno de ejecución
#   usando Runtime > Change Runtime Type > GPU
# assert len(tf.config.list_physical_devices('GPU')) > 0
# assert COMET_API_KEY != "", "Please insert your Comet API Key"

## 2.2 Conjunto de datos

![¡A bailar!](http://33.media.tumblr.com/3d223954ad0a77f4e98a7b87136aa395/tumblr_nlct5lFVbF1qhu7oio1_500.gif)

Hemos reunido un conjunto de datos de miles de canciones folclóricas irlandesas, representadas en notación ABC. Descarguemos el conjunto de datos e inspeccionémoslo:

In [ ]:
# Descargar el conjunto de datos
songs = mdl.lab1.load_training_data()

# ¡Imprimir una de las canciones para inspeccionarla con más detalle!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

Podemos convertir fácilmente una canción en notación ABC a una forma de onda de audio y reproducirla. Ten paciencia con esta conversión, puede tardar un poco.

In [ ]:
# Convertir la notación ABC a archivo de audio y escucharlo
mdl.lab1.play_song(example_song)

Una cosa importante a considerar es que esta notación musical no solo contiene información sobre las notas que se tocan, sino que además hay metainformación como el título de la canción, la tonalidad y el tempo. ¿Cómo afecta la cantidad de caracteres diferentes presentes en el archivo de texto a la complejidad del problema de aprendizaje? Esto será importante pronto, cuando generemos una representación numérica para los datos de texto.

In [ ]:
# Unir nuestra lista de cadenas de canciones en una sola cadena que contiene todas las canciones
songs_joined = "\n\n".join(songs)

# Encontrar todos los caracteres únicos en la cadena unida
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## 2.3 Procesar el conjunto de datos para la tarea de aprendizaje

Demos un paso atrás y consideremos nuestra tarea de predicción. Estamos tratando de entrenar un modelo RNN para aprender patrones en música ABC, y luego usar este modelo para generar (es decir, predecir) una nueva pieza musical basada en esta información aprendida.

Desglosando esto, lo que realmente le estamos pidiendo al modelo es: dado un carácter, o una secuencia de caracteres, ¿cuál es el siguiente carácter más probable? Entrenaremos el modelo para realizar esta tarea.

Para lograr esto, alimentaremos una secuencia de caracteres al modelo y lo entrenaremos para predecir la salida, es decir, el siguiente carácter en cada paso de tiempo. Las RNNs mantienen un estado interno que depende de los elementos vistos anteriormente, por lo que la información sobre todos los caracteres vistos hasta un momento dado se tendrá en cuenta al generar la predicción.

### Vectorizar el texto

Antes de comenzar a entrenar nuestro modelo RNN, necesitaremos crear una representación numérica de nuestro conjunto de datos basado en texto. Para hacer esto, generaremos dos tablas de búsqueda: una que mapea caracteres a números, y una segunda que mapea números de vuelta a caracteres. Recordemos que acabamos de identificar los caracteres únicos presentes en el texto.

In [ ]:
### Definir representación numérica del texto ###

# Crear un mapeo de carácter a índice único.
# Por ejemplo, para obtener el índice del carácter "d",
#   podemos evaluar `char2idx["d"]`.
char2idx = {u:i for i, u in enumerate(vocab)}

# Crear un mapeo de índices a caracteres. Este es
#   el inverso de char2idx y nos permite convertir de vuelta
#   desde un índice único al carácter en nuestro vocabulario.
idx2char = np.array(vocab)

Esto nos da una representación entera para cada carácter. Observa que los caracteres únicos (es decir, nuestro vocabulario) en el texto se mapean como índices de 0 a `len(unique)`. Echemos un vistazo a esta representación numérica de nuestro conjunto de datos:

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
### Vectorizar la cadena de canciones ###

'''POR HACER: Escribe una función para convertir la cadena de todas las canciones a una representación
    vectorizada (es decir, numérica). Usa el mapeo apropiado
    de arriba para convertir de caracteres del vocabulario a los índices correspondientes.

  NOTA: la salida de la función `vectorize_string`
  debe ser un np.array con `N` elementos, donde `N` es
  el número de caracteres en la cadena de entrada
'''
def vectorize_string(string):
  '''POR HACER'''

vectorized_songs = vectorize_string(songs_joined)

También podemos ver cómo se mapea la primera parte del texto a una representación entera:

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# verificar que vectorized_songs es un arreglo numpy
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Crear ejemplos de entrenamiento y objetivos

Nuestro siguiente paso es dividir el texto en secuencias de ejemplo que usaremos durante el entrenamiento. Cada secuencia de entrada que alimentemos a nuestra RNN contendrá `seq_length` caracteres del texto. También necesitaremos definir una secuencia objetivo para cada secuencia de entrada, que se usará para entrenar la RNN a predecir el siguiente carácter. Para cada entrada, el objetivo correspondiente contendrá la misma longitud de texto, pero desplazado un carácter a la derecha.

Para hacer esto, dividiremos el texto en fragmentos de `seq_length+1`. Supongamos que `seq_length` es 4 y nuestro texto es "Hello". Entonces, nuestra secuencia de entrada es "Hell" y la secuencia objetivo es "ello".

El método de lotes nos permitirá entonces convertir este flujo de índices de caracteres en secuencias del tamaño deseado.

In [ ]:
### Definición de lotes para crear ejemplos de entrenamiento ###

def get_batch(vectorized_songs, seq_length, batch_size):
  # la longitud de la cadena de canciones vectorizada
  n = vectorized_songs.shape[0] - 1
  # elegir aleatoriamente los índices de inicio para los ejemplos en el lote de entrenamiento
  idx = np.random.choice(n-seq_length, batch_size)

  '''POR HACER: construir una lista de secuencias de entrada para el lote de entrenamiento'''
  input_batch = # POR HACER

  '''POR HACER: construir una lista de secuencias de salida para el lote de entrenamiento'''
  output_batch = # POR HACER

  # x_batch, y_batch proporcionan las entradas y objetivos reales para el entrenamiento de la red
  x_batch = np.reshape(input_batch, [batch_size, seq_length])
  y_batch = np.reshape(output_batch, [batch_size, seq_length])
  return x_batch, y_batch


# ¡Realizar algunas pruebas simples para asegurarse de que la función de lotes funciona correctamente!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

Para cada uno de estos vectores, cada índice se procesa en un solo paso de tiempo. Así, para la entrada en el paso de tiempo 0, el modelo recibe el índice del primer carácter en la secuencia, e intenta predecir el índice del siguiente carácter. En el siguiente paso de tiempo, hace lo mismo, pero la RNN considera la información del paso anterior, es decir, su estado actualizado, además de la entrada actual.

Podemos hacer esto concreto observando cómo funciona esto sobre los primeros caracteres de nuestro texto:

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(np.squeeze(x_batch), np.squeeze(y_batch))):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

## 2.4 El modelo de Red Neuronal Recurrente (RNN)

Ahora estamos listos para definir y entrenar un modelo RNN en nuestro conjunto de datos de música ABC, y luego usar ese modelo entrenado para generar una nueva canción. Entrenaremos nuestra RNN usando lotes de fragmentos de canciones de nuestro conjunto de datos, que generamos en la sección anterior.

El modelo se basa en la arquitectura LSTM, donde usamos un vector de estado para mantener información sobre las relaciones temporales entre caracteres consecutivos. La salida final del LSTM se alimenta luego a una capa totalmente conectada [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) donde produciremos un softmax sobre cada carácter del vocabulario, y luego muestrearemos de esta distribución para predecir el siguiente carácter.

Como introdujimos en la primera parte de este laboratorio, usaremos la API de Keras, específicamente, [`tf.keras.Sequential`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential), para definir el modelo. Se utilizan tres capas para definir el modelo:

* [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding): Esta es la capa de entrada, que consiste en una tabla de búsqueda entrenable que mapea los números de cada carácter a un vector con `embedding_dim` dimensiones.
* [`tf.keras.layers.LSTM`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM): Nuestra red LSTM, con tamaño `units=rnn_units`.
* [`tf.keras.layers.Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense): La capa de salida, con `vocab_size` salidas.


<img src="https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/lstm_unrolled-01-01.png" alt="Drawing"/>

### Definir el modelo RNN

Ahora, definiremos una función que usaremos para construir el modelo.

In [ ]:
def LSTM(rnn_units):
  return tf.keras.layers.LSTM(
    rnn_units,
    return_sequences=True,
    recurrent_initializer='glorot_uniform',
    recurrent_activation='sigmoid',
    stateful=True,
  )

¡Ha llegado el momento! Completa los `TODOs` para definir el modelo RNN dentro de la función `build_model`, y luego llama a la función que acabas de definir para instanciar el modelo.

In [ ]:
### Definición del modelo RNN ###

'''POR HACER: Añadir capas LSTM y Dense para definir el modelo RNN usando la API Sequential.'''
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
  model = tf.keras.Sequential([
    # Capa 1: Capa de embedding para transformar índices en vectores densos
    #   de un tamaño de embedding fijo
    tf.keras.layers.Embedding(vocab_size, embedding_dim),

    # Capa 2: LSTM con `rnn_units` número de unidades.
    # POR HACER: Llama a la función LSTM definida arriba para añadir esta capa.
    LSTM('''POR HACER'''),

    # Capa 3: Capa Dense (totalmente conectada) que transforma la salida del LSTM
    #   al tamaño del vocabulario.
    # POR HACER: Añadir la capa Dense.
    '''POR HACER: CAPA DENSE AQUÍ'''
  ])

  return model

# Construir un modelo simple con hiperparámetros por defecto. Tendrás la
#   oportunidad de cambiarlos más adelante.
model = build_model(len(vocab), embedding_dim=256, rnn_units=1024, batch_size=32)
model.build(tf.TensorShape([32, 100]))  # [batch_size, sequence_length]

### Probar el modelo RNN

Siempre es buena idea ejecutar algunas verificaciones simples en nuestro modelo para ver que se comporta como se espera.  

Primero, podemos usar la función `Model.summary` para imprimir un resumen del funcionamiento interno de nuestro modelo. Aquí podemos verificar las capas del modelo, la forma de la salida de cada una de las capas, el tamaño del lote, etc.

In [ ]:
model.summary()

También podemos verificar rápidamente la dimensionalidad de nuestra salida, usando una longitud de secuencia de 100. Ten en cuenta que el modelo puede ejecutarse con entradas de cualquier longitud.

In [ ]:
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

### Predicciones del modelo no entrenado

Echemos un vistazo a lo que predice nuestro modelo no entrenado.

Para obtener predicciones reales del modelo, muestreamos de la distribución de salida, que se define por un `softmax` sobre nuestro vocabulario de caracteres. Esto nos dará índices de caracteres reales. Esto significa que estamos usando una [distribución categórica](https://en.wikipedia.org/wiki/Categorical_distribution) para muestrear sobre la predicción de ejemplo. Esto da una predicción del siguiente carácter (específicamente su índice) en cada paso de tiempo.

Nota que aquí muestreamos de esta distribución de probabilidad, en lugar de simplemente tomar el `argmax`, lo cual puede causar que el modelo se quede atrapado en un bucle.

Intentemos este muestreo para el primer ejemplo del lote.

In [ ]:
sampled_indices = tf.random.categorical(pred[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()
sampled_indices

Ahora podemos decodificar estos para ver el texto predicho por el modelo no entrenado:

In [ ]:
print("Input: \n", repr("".join(idx2char[x[0]])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices])))

Como puedes ver, ¡el texto predicho por el modelo no entrenado no tiene mucho sentido! ¿Cómo podemos mejorar? ¡Podemos entrenar la red!

## 2.5 Entrenamiento del modelo: pérdida y operaciones de entrenamiento

¡Ahora es el momento de entrenar el modelo!

En este punto, podemos pensar en nuestro problema de predicción del siguiente carácter como un problema estándar de clasificación. Dado el estado anterior de la RNN, así como la entrada en un paso de tiempo dado, queremos predecir la clase del siguiente carácter, es decir, predecir realmente el siguiente carácter.

Para entrenar nuestro modelo en esta tarea de clasificación, podemos usar una forma de la pérdida de `crossentropy` (pérdida de log-verosimilitud negativa). Específicamente, usaremos la pérdida [`sparse_categorical_crossentropy`](https://www.tensorflow.org/api_docs/python/tf/keras/losses/sparse_categorical_crossentropy), ya que utiliza objetivos enteros para tareas de clasificación categórica. Querremos calcular la pérdida usando los objetivos reales -- las `labels` -- y los objetivos predichos -- los `logits`.

Primero calculemos la pérdida usando nuestras predicciones de ejemplo del modelo no entrenado:

In [ ]:
### Definición de la función de pérdida ###

'''POR HACER: definir la función de pérdida para calcular y devolver la pérdida entre
    las etiquetas reales y las predicciones (logits). Establece el argumento from_logits=True.'''
def compute_loss(labels, logits):
  loss = tf.keras.losses.sparse_categorical_crossentropy('''POR HACER''', '''POR HACER''', from_logits=True) # POR HACER
  return loss

'''POR HACER: calcular la pérdida usando los verdaderos siguientes caracteres del lote de ejemplo
    y las predicciones del modelo no entrenado varias celdas arriba'''
example_batch_loss = compute_loss('''POR HACER''', '''POR HACER''') # POR HACER

print("Prediction shape: ", pred.shape, " # (batch_size, sequence_length, vocab_size)")
print("scalar_loss:      ", example_batch_loss.numpy().mean())

Comencemos definiendo algunos hiperparámetros para entrenar el modelo. Para empezar, hemos proporcionado algunos valores razonables para algunos de los parámetros. ¡Depende de ti usar lo que hemos aprendido en clase para ayudar a optimizar la selección de parámetros aquí!

In [ ]:
### Configuración de hiperparámetros y optimización ###

vocab_size = len(vocab)

# Parámetros del modelo:
params = dict(
  num_training_iterations = 3000,  # Aumentar esto para entrenar más tiempo
  batch_size = 8,  # Experimentar entre 1 y 64
  seq_length = 100,  # Experimentar entre 50 y 500
  learning_rate = 5e-3,  # Experimentar entre 1e-5 y 1e-1
  embedding_dim = 256,
  rnn_units = 1024,  # Experimentar entre 1 y 2048
)

# Ubicación del punto de control:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt.weights.h5")
os.makedirs(checkpoint_dir, exist_ok=True)

Habiendo definido nuestros hiperparámetros, podemos configurar el seguimiento de experimentos con Comet. Los [`Experiment`](https://www.comet.com/docs/v2/api-and-sdk/python-sdk/reference/Experiment/) son los objetos centrales en Comet y nos permitirán rastrear el entrenamiento y el desarrollo del modelo. Aquí hemos escrito una función corta para crear un nuevo experimento de Comet. Ten en cuenta que en esta configuración, cuando los hiperparámetros cambien, puedes ejecutar la función `create_experiment()` para iniciar un nuevo experimento. Todos los experimentos definidos con el mismo `project_name` estarán bajo ese proyecto en tu interfaz de Comet.



In [ ]:
### Crear un experimento de Comet para rastrear nuestra ejecución de entrenamiento ###

def create_experiment():
  # finalizar cualquier experimento anterior
  if 'experiment' in locals():
    experiment.end()

  # iniciar el experimento de Comet para seguimiento
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name="6S191_Lab1_Part2")
  # registrar nuestros hiperparámetros, definidos arriba, en el experimento
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

Ahora, estamos listos para definir nuestra operación de entrenamiento -- el optimizador y la duración del entrenamiento -- y usar esta función para entrenar el modelo. Experimentarás con la elección del optimizador y la duración del entrenamiento de tus modelos, y verás cómo estos cambios afectan la salida de la red. Algunos optimizadores que podrías probar son [`Adam`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam?version=stable) y [`Adagrad`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adagrad?version=stable).

Primero, instanciaremos un nuevo modelo y un optimizador. Luego, usaremos el método [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape) para realizar las operaciones de retropropagación.

También generaremos una impresión del progreso del modelo durante el entrenamiento, lo que nos ayudará a visualizar fácilmente si estamos minimizando la pérdida o no.

In [ ]:
### Definir optimizador y operación de entrenamiento ###

'''POR HACER: instanciar un nuevo modelo para entrenamiento usando la función `build_model`
  y los hiperparámetros creados arriba.'''
model = build_model('''POR HACER: argumentos''')

'''POR HACER: instanciar un optimizador con su tasa de aprendizaje.
  Consulta el sitio web de TensorFlow para ver una lista de optimizadores soportados.
  https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/
  Intenta usar el optimizador Adam para empezar.'''
optimizer = # POR HACER

@tf.function
def train_step(x, y):
  # Usar tf.GradientTape()
  with tf.GradientTape() as tape:

    '''POR HACER: alimentar la entrada actual al modelo y generar predicciones'''
    y_hat = model('''POR HACER''')

    '''POR HACER: ¡calcular la pérdida!'''
    loss = compute_loss('''POR HACER''', '''POR HACER''')

  # Ahora, calcular los gradientes
  '''POR HACER: completar la llamada a la función para el cálculo del gradiente.
      Recuerda que queremos el gradiente de la pérdida con respecto a todos
      los parámetros del modelo.
      PISTA: usa `model.trainable_variables` para obtener una lista de todos los
      parámetros del modelo.'''
  grads = tape.gradient('''POR HACER''', '''POR HACER''')

  # Aplicar los gradientes al optimizador para que pueda actualizar el modelo
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  return loss

##################
# ¡Comenzar el entrenamiento!#
##################

history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
experiment = create_experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # limpiar si existe
for iter in tqdm(range(params["num_training_iterations"])):

  # Tomar un lote y propagarlo a través de la red
  x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])
  loss = train_step(x_batch, y_batch)

  # ¡registrar la pérdida en la interfaz de Comet! podremos rastrearla allí.
  experiment.log_metric("loss", loss.numpy().mean(), step=iter)
  # Actualizar la barra de progreso y también visualizar dentro del notebook
  history.append(loss.numpy().mean())
  plotter.plot(history)

  # ¡Actualizar el modelo con los pesos cambiados!
  if iter % 100 == 0:
    model.save_weights(checkpoint_prefix)

# Guardar el modelo entrenado y los pesos
model.save_weights(checkpoint_prefix)
experiment.flush()


## 2.6 Generar música usando el modelo RNN

¡Ahora podemos usar nuestro modelo RNN entrenado para generar algo de música! Al generar música, tendremos que alimentar al modelo algún tipo de semilla para que comience (¡porque no puede predecir nada sin algo con qué empezar!).

Una vez que tengamos una semilla generada, podemos entonces predecir iterativamente cada carácter sucesivo (recuerda, estamos usando la representación ABC para nuestra música) usando nuestra RNN entrenada. Más específicamente, recordemos que nuestra RNN produce un `softmax` sobre posibles caracteres sucesivos. Para la inferencia, muestreamos iterativamente de estas distribuciones, y luego usamos nuestras muestras para codificar una canción generada en el formato ABC.

¡Luego, todo lo que tenemos que hacer es escribirlo en un archivo y escuchar!


### Restaurar el último punto de control

Para mantener este paso de inferencia simple, usaremos un tamaño de lote de 1. Debido a cómo se pasa el estado de la RNN de un paso de tiempo a otro, el modelo solo podrá aceptar un tamaño de lote fijo una vez que se construya.

Para ejecutar el modelo con un `batch_size` diferente, necesitaremos reconstruir el modelo y restaurar los pesos desde el último punto de control, es decir, los pesos después del último punto de control durante el entrenamiento:

In [ ]:
'''POR HACER: Reconstruir el modelo usando batch_size=1'''
model = build_model('''POR HACER''', '''POR HACER''', '''POR HACER''', batch_size=1)

# Restaurar los pesos del modelo del último punto de control después del entrenamiento
model.build(tf.TensorShape([1, None]))
model.load_weights(checkpoint_prefix)

model.summary()

Observa que hemos alimentado un `batch_size` fijo de 1 para la inferencia.

### El procedimiento de predicción

Ahora, estamos listos para escribir el código para generar texto en formato de música ABC:

* Inicializar una cadena de inicio "semilla" y el estado de la RNN, y establecer el número de caracteres que queremos generar.

* Usar la cadena de inicio y el estado de la RNN para obtener la distribución de probabilidad sobre el siguiente carácter predicho.

* Muestrear de la distribución multinomial para calcular el índice del carácter predicho. Este carácter predicho se usa luego como la siguiente entrada al modelo.

* En cada paso de tiempo, el estado actualizado de la RNN se retroalimenta al modelo, de modo que ahora tiene más contexto para hacer la siguiente predicción. Después de predecir el siguiente carácter, los estados actualizados de la RNN se retroalimentan nuevamente al modelo, que es cómo aprende las dependencias secuenciales en los datos, ya que obtiene más información de las predicciones anteriores.

![Inferencia LSTM](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/lstm_inference.png)

Completa y experimenta con este bloque de código (¡así como con algunos aspectos de la definición de la red y el entrenamiento!), y observa cómo se desempeña el modelo. ¿Cómo se comparan las canciones generadas después de entrenar con un número pequeño de épocas con las generadas después de una mayor duración de entrenamiento?

In [ ]:
### Predicción de una canción generada ###

def generate_text(model, start_string, generation_length=1000):
  # Paso de evaluación (generando texto ABC usando el modelo RNN aprendido)

  '''POR HACER: convertir la cadena de inicio a números (vectorizar)'''
  input_eval = ['''POR HACER''']
  input_eval = tf.expand_dims(input_eval, 0)

  # Cadena vacía para almacenar nuestros resultados
  text_generated = []

  # Aquí el tamaño del lote == 1
  for layer in model.layers:
    if hasattr(layer, "reset_states"):
        layer.reset_states()
  tqdm._instances.clear()

  for i in tqdm(range(generation_length)):
      '''POR HACER: evaluar las entradas y generar las predicciones del siguiente carácter'''
      predictions = model('''POR HACER''')

      # Eliminar la dimensión del lote
      predictions = tf.squeeze(predictions, 0)

      '''POR HACER: usar una distribución multinomial para muestrear'''
      predicted_id = tf.random.categorical('''POR HACER''', num_samples=1)[-1,0].numpy()

      # Pasar la predicción junto con el estado oculto anterior
      #   como las siguientes entradas al modelo
      input_eval = tf.expand_dims([predicted_id], 0)

      '''POR HACER: ¡añadir el carácter predicho al texto generado!'''
      # Pista: considera en qué formato está la predicción vs. la salida
      text_generated.append('''POR HACER''')

  return (start_string + ''.join(text_generated))

In [ ]:
'''POR HACER: ¡Usa el modelo y la función definida arriba para generar texto en formato ABC de longitud 1000!
    Como puedes notar, los archivos ABC comienzan con "X" - esta puede ser una buena cadena de inicio.'''
generated_text = generate_text('''POR HACER''', start_string="X", generation_length=1000)

### ¡Reproducir la música generada!

¡Ahora podemos llamar a una función para convertir el texto en formato ABC a un archivo de audio, y luego reproducirlo para escuchar nuestra música generada! Intenta entrenar más tiempo si la canción resultante no es lo suficientemente larga, ¡o regenera la canción!

Guardaremos la canción en Comet -- podrás encontrar tus canciones en las páginas `Audio` y `Assets & Artifacts` en tu interfaz de Comet para el proyecto. Consulta la documentación de [`log_asset()`](https://www.comet.com/docs/v2/api-and-sdk/python-sdk/reference/Experiment/#experimentlog_asset), donde verás cómo especificar nombres de archivo y otros parámetros para guardar tus recursos.

In [ ]:
### Reproducir canciones generadas ###

generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # Sintetizar la forma de onda de una canción
  waveform = mdl.lab1.play_song(song)

  # Si es una canción válida (sintaxis correcta), ¡reproduzcámosla!
  if waveform:
    print("Generated song", i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # guardar tu canción en la interfaz de Comet -- puedes acceder a ella allí
    experiment.log_asset(wav_file_path)

In [ ]:
# cuando termines, finalizar el experimento de Comet
experiment.end()

## 2.7 ¡Experimenta y **obtén premios por las mejores canciones**!

¡Felicidades por crear tu primer modelo de secuencias en TensorFlow! Es un logro bastante grande, y esperamos que tengas algunas melodías geniales para demostrarlo.

Considera cómo podrías mejorar tu modelo y qué parece ser lo más importante en términos de rendimiento. Aquí hay algunas ideas para comenzar:

*  ¿Cómo afecta el número de épocas de entrenamiento al rendimiento?
*  ¿Qué pasa si alteras o aumentas el conjunto de datos?
*  ¿La elección de la cadena de inicio afecta significativamente el resultado?

¡Intenta optimizar tu modelo y envía tu mejor canción! **Los participantes serán elegibles para premios durante la oferta de enero de 2025. Para participar en la competencia, debes subir lo siguiente a [este enlace de envío](https://www.dropbox.com/request/4hqfsOnLtX4jH1W3ynfp):**

* una grabación de tu canción;
* un notebook de iPython con el código que usaste para generar la canción;
* una descripción y/o diagrama de la arquitectura e hiperparámetros que usaste -- si hay alguna modificación adicional o interesante que hayas hecho al código de plantilla, por favor inclúyela en tu descripción.

**Nombra tu archivo en el siguiente formato: ``[Nombre]_[Apellido]_RNNMusic``, seguido del formato de archivo (.zip, .mp4, .ipynb, .pdf, etc). Se prefieren archivos ZIP de los tres componentes sobre archivos individuales. Si envías archivos individuales, debes nombrar los archivos individuales según la nomenclatura anterior.**

También puedes tuitearnos a [@MITDeepLearning](https://twitter.com/MITDeepLearning) una copia de la canción (¡pero esto no te inscribirá en la competencia!). Mira esta canción de ejemplo generada por una estudiante anterior (crédito Ana Heart): <a href="https://twitter.com/AnaWhatever16/status/1263092914680410112?s=20">canción del 20 de mayo de 2020.</a>
<script async src="https://platform.twitter.com/widgets.js" charset="utf-8"></script>

¡Diviértete y feliz escucha!

![¡A bailar!](http://33.media.tumblr.com/3d223954ad0a77f4e98a7b87136aa395/tumblr_nlct5lFVbF1qhu7oio1_500.gif)
